## Graph E2E 테스트

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.03.20 </div>
<div style="text-align: right"> last update : 2026.03.25 </div>

개정 이력  
- `2026.03.20` : 노트북 최초 작성
- `2026.03.25` : 2-Phase 그래프 토폴로지 반영 (Phase 1: resource_estimation + similar_project → phase1_sync → Phase 2: scoring + risk_analysis → final_verdict), 노드 목록/엣지 구조 업데이트, 스트리밍 Phase별 이벤트 순서 검증, risk_interdependencies 출력 검증, 실행 시간 측정 추가

**사전 요구사항:**
- PostgreSQL 실행 (`make docker-up && make migrate && make seed`)
- `.env` 파일에 API 키 설정 (OpenAI/Anthropic, Pinecone)
- LangGraph 그래프 전체를 End-to-End로 실행

**현재 그래프 토폴로지:**
```
deal_structuring → [conditional]
  ├── hold_verdict → END
  └── Phase 1 [parallel]: resource_estimation + similar_project
       → phase1_sync
       → Phase 2 [parallel]: scoring + risk_analysis
       → final_verdict → END
```

In [ ]:
import sys

sys.path.insert(0, "../../")

from dotenv import load_dotenv

load_dotenv()

### 1. 그래프 빌드

In [ ]:
from backend.app.agent.graph import build_graph

graph = build_graph()
print(f"Graph type: {type(graph).__name__}")

In [ ]:
# 노드 및 엣지 구조 확인 — 2-Phase 토폴로지 검증
graph_def = graph.get_graph()

print("=== Nodes ===")
for node in graph_def.nodes:
    print(f"  - {node}")

# phase1_sync 노드 존재 확인
node_names = [str(n) for n in graph_def.nodes]
assert "phase1_sync" in node_names, "phase1_sync 노드가 누락됨!"
print("\n✓ phase1_sync 노드 확인 완료")

print("\n=== Edges ===")
for edge in graph_def.edges:
    print(f"  {edge}")

print("\n=== 기대 토폴로지 ===")
print("  deal_structuring → conditional_edges")
print("    ├── hold_verdict → END")
print("    └── resource_estimation + similar_project (Phase 1)")
print("  resource_estimation → phase1_sync")
print("  similar_project → phase1_sync")
print("  phase1_sync → scoring + risk_analysis (Phase 2)")
print("  scoring → final_verdict")
print("  risk_analysis → final_verdict")
print("  final_verdict → END")

### 2. 그래프 시각화

In [ ]:
# Mermaid 다이어그램 출력
mermaid = graph_def.draw_mermaid()
print(mermaid)

In [ ]:
# Jupyter에서 Mermaid 렌더링 (IPython 지원 시)
from IPython.display import Image, display

try:
    img_bytes = graph_def.draw_mermaid_png()
    display(Image(img_bytes))
except Exception as e:
    print(f"PNG 렌더링 불가 (mermaid-cli 필요): {e}")
    print("위 Mermaid 코드를 https://mermaid.live 에 붙여넣어 확인 가능")

### 3. 정상 딜 E2E 실행

In [ ]:
import uuid

# 충분한 정보가 포함된 정상 딜 입력 — [딜 기본 정보] + [상세 내용] 형식
normal_deal_input = """[딜 기본 정보]
고객사: 메가테크
고객 규모: 대기업
산업군: 제조업
프로젝트명: 공장 설비 예지보전 AI 시스템 개발
예상 금액: 20000만원
금액 단위: 만원
기간: 5개월

[상세 내용]
## 프로젝트 배경
기존 정기 점검 체계에서 AI 기반 예측 유지보수로 전환하여 설비 가동률 향상 및 유지보수 비용 절감을 목표로 함.

## 기술 요구사항
- Python, TensorFlow 기반 시계열 예측 모델
- IoT 센서 데이터 실시간 수집/처리 파이프라인
- 대시보드 (React 기반)
- MLOps: 모델 재학습 자동화 파이프라인

## 보안 요구사항
- 공장 내 보안 네트워크 환경에서만 운영, 온프레미스 배포 필수

## 결제 조건
착수금 30%, 중간 40%, 완료 30%

## 이해관계자
- 발주측 PM: 설비관리팀 김부장
- 의사결정권자: 제조본부장"""

test_deal_id = str(uuid.uuid4())
print(f"Deal ID: {test_deal_id}")

In [ ]:
# graph.ainvoke로 전체 파이프라인 실행
result = await graph.ainvoke({
    "deal_id": test_deal_id,
    "deal_input": normal_deal_input,
})

print(f"실행 완료! Result keys: {list(result.keys())}")

### 4. 결과 분석

In [ ]:
import json

print("=== 핵심 결과 ===")
print(f"Status: {result.get('status')}")
print(f"Verdict: {result.get('verdict')}")
print(f"Total Score: {result.get('total_score')}")
print(f"Errors: {result.get('errors', [])}")

In [ ]:
# structured_deal 확인
print("=== Structured Deal ===")
print(json.dumps(result.get("structured_deal", {}), ensure_ascii=False, indent=2))

In [ ]:
# scores 확인 — 서버사이드 재계산 적용 여부 검증
print("=== Scores ===")
for s in result.get("scores", []):
    print(f"  {s['criterion']}: {s['score']}점 × {s['weight']} = {s['weighted_score']} — {s['rationale'][:60]}...")
print(f"\n  총점: {result.get('total_score')} → {result.get('verdict')}")

# verdict 임계값 검증
ts = result.get("total_score", 0)
v = result.get("verdict", "")
if ts >= 70:
    expected_v = "go"
elif ts >= 40:
    expected_v = "conditional_go"
else:
    expected_v = "no_go"
print(f"  verdict 검증: {ts}점 → 기대={expected_v}, 실제={v} {'✓' if v == expected_v else '✗'}")

In [ ]:
# resource_estimate 상세 확인
print("=== Resource Estimate ===")
re = result.get("resource_estimate", {})
print(f"duration_months: {re.get('duration_months')}")
print(f"duration_with_buffer: {re.get('duration_with_buffer')}")
print(f"risk_buffer_ratio: {re.get('risk_buffer_ratio')}")

if "cost_breakdown" in re:
    cb = re["cost_breakdown"]
    print(f"cost_breakdown: labor={cb.get('labor_cost')}만원, overhead={cb.get('overhead_cost')}만원, total={cb.get('total_cost')}만원")

if "profitability" in re:
    prof = re["profitability"]
    print(f"profitability: deal_amount={prof.get('deal_amount')}만원, cost={prof.get('estimated_cost')}만원, margin={prof.get('expected_margin')}")

print(f"\nwork_breakdown: {len(re.get('work_breakdown', []))}건")
print(f"phases: {len(re.get('phases', []))}건")
print(f"team_composition: {len(re.get('team_composition', []))}건")

In [ ]:
# risks + risk_interdependencies 확인
print("=== Risks ===")
for r in result.get("risks", []):
    print(f"  [{r.get('level')}] {r.get('category')}: {r.get('item')} — {r.get('description', '')[:60]}...")
    if r.get("mitigation"):
        print(f"    → 완화: {r.get('mitigation')[:60]}...")

# risk_interdependencies (신규)
interdeps = result.get("risk_interdependencies", [])
print(f"\n=== Risk Interdependencies ({len(interdeps)}건) ===")
for ri in interdeps:
    print(f"  {ri.get('risk_pair', [])} → {ri.get('combined_effect', '')}")

In [ ]:
# similar_projects 확인 — relevance_score 포함
print("=== Similar Projects ===")
similar = result.get("similar_projects", [])
if similar:
    for p in similar:
        print(f"  ✓ {p.get('project_name')}: similarity={p.get('similarity_score')}, relevance={p.get('relevance_score')}")
        if p.get("lessons_learned"):
            print(f"    교훈: {p.get('lessons_learned')[:80]}...")
else:
    print("유사 프로젝트 없음")

In [ ]:
# final_report 마크다운 렌더링
from IPython.display import Markdown, display

display(Markdown(result.get("final_report", "리포트 없음")))

### 5. Hold 케이스 테스트

In [ ]:
# 정보가 부족한 딜 입력 → Hold 경로 확인
hold_deal_input = "AI 프로젝트를 하고 싶습니다. 예산이나 기간은 아직 미정입니다."

hold_deal_id = str(uuid.uuid4())
print(f"Hold Deal ID: {hold_deal_id}")

In [ ]:
hold_result = await graph.ainvoke({
    "deal_id": hold_deal_id,
    "deal_input": hold_deal_input,
})

print("=== Hold 케이스 결과 ===")
print(f"Status: {hold_result.get('status')}")
print(f"Verdict: {hold_result.get('verdict')}")  # "pending" (hold 경로)
print(f"Total Score: {hold_result.get('total_score')}")
print(f"Missing Fields: {hold_result.get('structured_deal', {}).get('missing_fields', [])}")
print(f"\nFinal Report:\n{hold_result.get('final_report', '')}")

# Hold 검증
assert hold_result.get("verdict") == "pending", f"Hold 경로에서 verdict가 'pending'이 아님: {hold_result.get('verdict')}"
assert hold_result.get("total_score") == 0.0, f"Hold 경로에서 total_score가 0이 아님: {hold_result.get('total_score')}"
print("\n✓ Hold 케이스 검증 통과")

### 6. 스트리밍 실행 (단계별 진행 확인)

In [ ]:
import time

# graph.astream으로 노드별 실행 결과를 실시간 확인 — Phase별 이벤트 순서 검증
stream_deal_id = str(uuid.uuid4())

print("=== 스트리밍 실행 시작 ===")
accumulated = {}
node_order = []
phase_times = {}
start_time = time.time()

PHASE1_NODES = {"resource_estimation", "similar_project"}
PHASE2_NODES = {"scoring", "risk_analysis"}

async for event in graph.astream(
    {"deal_id": stream_deal_id, "deal_input": normal_deal_input},
    stream_mode="updates",
):
    for node_name, node_output in event.items():
        elapsed = round(time.time() - start_time, 1)
        node_order.append(node_name)

        # Phase 전환 표시
        if node_name in PHASE1_NODES and "=== Phase 1" not in str(phase_times):
            if not phase_times.get("phase1_start"):
                phase_times["phase1_start"] = elapsed
                print(f"\n{'='*60}")
                print(f"=== Phase 1 시작 ({elapsed}s) ===")
                print(f"{'='*60}")
        if node_name == "phase1_sync":
            phase_times["phase1_end"] = elapsed
            print(f"\n{'='*60}")
            print(f"=== Phase 1 완료 → Phase 2 시작 ({elapsed}s) ===")
            print(f"{'='*60}")
        if node_name in PHASE2_NODES and not phase_times.get("phase2_start"):
            phase_times["phase2_start"] = elapsed

        print(f"\n--- [{node_name}] 완료 ({elapsed}s) ---")
        print(f"  Output keys: {list(node_output.keys())}")

        # 주요 필드 요약 출력
        if "structured_deal" in node_output:
            sd = node_output["structured_deal"]
            print(f"  customer: {sd.get('customer_name')}, industry: {sd.get('customer_industry')}")
            print(f"  missing_fields: {sd.get('missing_fields', [])}")
        if "verdict" in node_output:
            print(f"  verdict: {node_output['verdict']}, score: {node_output.get('total_score')}")
        if "resource_estimate" in node_output:
            re = node_output["resource_estimate"]
            print(f"  duration: {re.get('duration_months')}개월 (buffer: {re.get('duration_with_buffer')})")
            if "profitability" in re:
                print(f"  margin: {re['profitability'].get('expected_margin')}")
        if "risks" in node_output:
            print(f"  risks count: {len(node_output['risks'])}")
            print(f"  risk_interdependencies count: {len(node_output.get('risk_interdependencies', []))}")
        if "similar_projects" in node_output:
            print(f"  similar_projects count: {len(node_output['similar_projects'])}")
        if "final_report" in node_output:
            print(f"  final_report length: {len(node_output['final_report'])} chars")

        accumulated.update(node_output)

total_time = round(time.time() - start_time, 1)
phase_times["total"] = total_time

print(f"\n{'='*60}")
print(f"=== 스트리밍 실행 완료 ({total_time}s) ===")
print(f"{'='*60}")

# Phase별 실행 순서 검증
print("\n=== 실행 순서 검증 ===")
print(f"노드 실행 순서: {' → '.join(node_order)}")

# Phase 1 노드가 Phase 2 노드보다 먼저 실행되었는지 확인
phase1_indices = [node_order.index(n) for n in PHASE1_NODES if n in node_order]
phase2_indices = [node_order.index(n) for n in PHASE2_NODES if n in node_order]
if phase1_indices and phase2_indices:
    assert max(phase1_indices) < min(phase2_indices), "Phase 1 노드가 Phase 2 이후에 실행됨!"
    print("✓ Phase 1 → Phase 2 순서 보장 확인")

print("\n=== 실행 시간 ===")
for k, v in phase_times.items():
    print(f"  {k}: {v}s")

### 7. 에러 핸들링 테스트

In [ ]:
# 빈 입력으로 실행 → Hold 경로 또는 에러 핸들링 확인
error_deal_id = str(uuid.uuid4())

error_result = await graph.ainvoke({
    "deal_id": error_deal_id,
    "deal_input": "",
})

print("=== 빈 입력 결과 ===")
print(f"Status: {error_result.get('status')}")
print(f"Verdict: {error_result.get('verdict')}")
print(f"Total Score: {error_result.get('total_score')}")
print(f"Errors: {error_result.get('errors', [])}")
print(f"Final Report:\n{error_result.get('final_report', '')}")

# 에러 격리 확인 — 한 노드 실패가 전체를 중단하지 않음
print("\n=== 에러 격리 검증 ===")
print(f"status가 존재: {'✓' if error_result.get('status') else '✗'}")
print(f"final_report가 존재: {'✓' if error_result.get('final_report') else '✗'}")